# 🗣️ MCP로 받고 SQLite에 적재하기 — Instructor

에이전트가 받은 값을 코드로 남기고, 삼성전자의 재무 지표까지 확인한다.

> **🖥️ 환경 설정**
>
> - **로컬**: VS Code에서 이 노트북을 열고 `FD_26/.venv` 커널을 선택한다. 실행·설치는 모두 `FD_26`에서 한다.
>
> - **의존성**: `pandas`·`requests`·`python-dotenv`·`supabase`가 없으면 `FD_26` 루트에서 `uv add pandas requests python-dotenv supabase`를 한 번 실행한다(이미 있으면 건너뜀).
>
> - **기본 런타임**: VS Code Copilot Chat(에이전트 모드)이 기본이다. Claude Code·Codex·Agy를 이미 쓰고 있다면 그걸 그대로 쓴다.
>
> - **데이터**: 이 노트북은 같은 폴더의 `data/`에 있는 `market_data.db`를 사용한다.
>
> **⌨️ 단축키 안내**
>
> | 단축키 | 동작 |
> |--------|------|
> | `Shift + Enter` | 셀 실행 후 다음 셀 이동 |
> | `Esc → A` | 위에 셀 삽입 |
> | `Esc → B` | 아래에 셀 삽입 |
> | `Esc → DD` | 셀 삭제 |

In [1]:
# 🖥️ 환경 설정
# - VS Code에서 FD_26/.venv 커널을 선택했는지 먼저 확인한다 (실행·설치는 전부 FD_26에서).

import os
import sys

print(sys.executable)
# 예상 출력: (자기 프로젝트 경로)/FD_26/.venv/bin/python3


c:\Projects\FD_26\.venv\Scripts\python.exe


## 📥 자연어로 받고, 코드로 재현 가능하게 남긴다

[→ §500 강의노트](../notes/FD-04-500__load-and-measure.md)

§410에서 설치와 검증을 마쳤다면, 에이전트에게 받은 삼성전자 재무제표를 SQLite에 남긴다. 먼저 같은 종목·연도에 한 행만 남는 표를 준비한다.

In [2]:
# 📌 §500-1 📥 자연어로 받고, 코드로 재현 가능하게 남긴다
import sqlite3
import pandas as pd

con = sqlite3.connect("data/market_data.db")
existing_schema = con.execute(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='financials'"
).fetchone()

if existing_schema and "PRIMARY KEY" not in existing_schema[0]:
    with con:
        con.execute("ALTER TABLE financials RENAME TO financials_old")
        con.execute("""
        CREATE TABLE financials (
            code TEXT NOT NULL,
            year INTEGER NOT NULL,
            revenue INTEGER,
            operating_profit INTEGER,
            net_income INTEGER,
            equity INTEGER,
            PRIMARY KEY (code, year)
        )
        """)
        con.execute("""
        INSERT OR IGNORE INTO financials (code, year, revenue, operating_profit, net_income, equity)
        SELECT code, year, revenue, operating_profit, net_income, equity FROM financials_old
        """)
    copied = con.execute("SELECT COUNT(*) FROM financials").fetchone()[0]
    print(f"옛 표(financials_old)에서 {copied}행을 새 PK 표로 옮겼다 — 원본은 financials_old에 그대로 남아 있다.")
else:
    con.execute("""
    CREATE TABLE IF NOT EXISTS financials (
        code TEXT NOT NULL,
        year INTEGER NOT NULL,
        revenue INTEGER,
        operating_profit INTEGER,
        net_income INTEGER,
        equity INTEGER,
        PRIMARY KEY (code, year)
    )
    """)
    con.commit()

con.close()
# 예상 출력(PK 없는 옛 표였다면): 옛 표(financials_old)에서 24행을 새 PK 표로 옮겼다 — 원본은 financials_old에 그대로 남아 있다.
# 예상 출력(이미 PK가 있으면): (출력 없이 종료 — 아무 것도 바뀌지 않는다)

> **학생이 보내는 요청**
>
> "삼성전자(corp_code 00126380) 2025년 사업보고서 연결(CFS) 재무제표에서 매출액, 영업이익, 당기순이익, 자본총계를 가져와줘. 단위는 억원으로 알려줘."

같은 종목·연도를 다시 받아도 행이 늘지 않게 `ON CONFLICT`로 갱신한다. 에이전트의 답을 확인한 뒤, 단위가 억원인지 확인하고 딕셔너리에 옮긴다.
> **⏱️ 20분 지점 체크포인트**
>
> MCP가 안 되면 아래 삼성전자 2025 연결 기준값(억원, §500-3 예상 출력과 동일)으로 cell 5부터 계속한다: revenue 3336059 · operating_profit 436010 · net_income 452068(700에서 직접 받은 값과 같다) · equity 4363203(700과 동일).

In [3]:
# 📌 §500-2 📥 자연어로 받고, 코드로 재현 가능하게 남긴다
con = sqlite3.connect("data/market_data.db")

# opendart-mcp 자연어 요청 결과 — ①~③에서 직접 받아 확인한 값을 여기 옮겨 적는다 (단위: 억원)
mcp_result = {"code": "005930", "year": 2025, "revenue": 3_336_059,
              "operating_profit": 436_010, "net_income": 452_068, "equity": 4_363_203}

# 있으면 덮어쓰고(UPDATE) 없으면 추가한다 — 같은 종목·연도를 몇 번 다시 실행해도 행이 늘지 않는다
con.execute("""
INSERT INTO financials (code, year, revenue, operating_profit, net_income, equity)
VALUES (:code, :year, :revenue, :operating_profit, :net_income, :equity)
ON CONFLICT(code, year) DO UPDATE SET
    revenue=excluded.revenue, operating_profit=excluded.operating_profit,
    net_income=excluded.net_income, equity=excluded.equity
""", mcp_result)
con.commit()
con.close()
# 예상 출력: (에러 없이 종료)

## 🔍 적재됐는지 SELECT로 확인

[→ §500 강의노트](../notes/FD-04-500__load-and-measure.md)

두 번 실행해도 삼성전자 2025 행 수가 정확히 1인지 확인한다. 이것이 같은 키를 다시 받아도 한 행만 남는 멱등 적재다.

In [4]:
# 📌 §500-3 🔍 적재됐는지 SELECT로 확인
con = sqlite3.connect("data/market_data.db")
row = pd.read_sql("SELECT * FROM financials WHERE code='005930' AND year=2025", con)
print(row)
print("삼성전자 2025 행 수:", pd.read_sql(
    "SELECT COUNT(*) AS n FROM financials WHERE code='005930' AND year=2025", con
)["n"][0])
con.close()
# 예상 출력:      code  year   revenue  operating_profit  net_income   equity
# 예상 출력: 0  005930  2025  3336059            436010      452068  4363203
# 예상 출력: 삼성전자 2025 행이 정확히 1개(전체 행 수는 Week03 표를 옮겼으면 24, 새 DB면 1)

     code  year  revenue  operating_profit  net_income   equity
0  005930  2025  3336059            436010      452068  4363203
삼성전자 2025 행 수: 1


## 📐 시가총액을 구해 세 지표를 완성한다

[→ §500 강의노트](../notes/FD-04-500__load-and-measure.md)

적재한 재무 숫자와 최신 종가·보통주 주식수로 시가총액을 만든다. 모든 금액 단위를 억원으로 맞춘 뒤 PER·PBR·ROE를 읽는다.

In [5]:
# 📌 §500-4 📐 시가총액을 구해 세 지표를 완성한다
con = sqlite3.connect("data/market_data.db")
price_row = pd.read_sql(
    "SELECT close, date FROM returns WHERE code='005930' ORDER BY date DESC LIMIT 1", con
)
close_price, price_date = price_row["close"][0], price_row["date"][0]

fin_row = pd.read_sql(
    "SELECT net_income, equity FROM financials WHERE code='005930' AND year=2025", con
)
net_income_eok, equity_eok = fin_row["net_income"][0], fin_row["equity"][0]
con.close()

shares_outstanding = 5_919_637_922  # 삼성전자 보통주 발행주식총수 — DART `stockTotqySttus`(주식의총수현황) 2025 기준값(①의 요청에는 포함되지 않은 별도 출처)
market_cap_eok = close_price * shares_outstanding / 1e8  # 원 → 억원

per = market_cap_eok / net_income_eok
pbr = market_cap_eok / equity_eok
roe = net_income_eok / equity_eok

print(f"종가({price_date} 기준): {close_price:,}원 · 시가총액: {market_cap_eok:,.0f}억원")
print(f"PER: {per:.2f} · PBR: {pbr:.2f} · ROE: {roe*100:.1f}%")
# 예상 출력: 종가(returns 테이블의 최신 날짜 기준): 67,784원 · 시가총액: 4,012,567억원
# 예상 출력: PER: 8.88 · PBR: 0.92 · ROE: 10.4%

종가(2026-06-25 기준): 67,784원 · 시가총액: 4,012,567억원
PER: 8.88 · PBR: 0.92 · ROE: 10.4%


## 다음 단계

PER·PBR·ROE는 매수·매도 판정이 아니라 데이터를 읽는 렌즈다. 다음 노트북에서는 같은 `financials` 행을 Supabase 클라우드에도 올리고 조회한다.